<a href="https://colab.research.google.com/github/DS-Challenge-Hackathon-J-Group/DataScience-Hackathon/blob/main/Phase1_Understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RMDS Data Science Challenge 2026 – Dataset B (H1N1 vaccine)
## Phase 1: Competition & dataset understanding
**Purpose:** load the data, verify integrity (shapes, ID consistency, submission format, target encoding),
define feature groups from the data dictionary, and set up the **cost-based metric framework** and **experiment log**
that every later phase will reuse. No modelling happens here.

### 1.1 Setup & configuration

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED   = 42
TARGET = "h1n1_vaccine"      # confirmed from training CSV header
ID_COL = "respondent_id"

# Put the three CSVs in your team GitHub repo (e.g. <repo>/data/) and clone it into Colab,
# OR upload them to Colab. Edit DATA_DIR accordingly.
DATA_DIR = "data"
OUT_DIR  = "outputs"          # predictions, logs, configs are saved here
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

FILES = ["dataset_B_training.csv", "dataset_B_testing.csv", "dataset_B_example_submission.csv"]
if not all(os.path.exists(f"{DATA_DIR}/{f}") for f in FILES):
    try:
        from google.colab import files          # fallback: manual upload in Colab
        print("Upload the 3 CSV files:")
        for name, content in files.upload().items():
            open(f"{DATA_DIR}/{name}", "wb").write(content)
    except ImportError:
        raise FileNotFoundError(f"Place {FILES} in '{DATA_DIR}/'")
print("Files found.")

### 1.2 Load data and run integrity checks

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/dataset_B_training.csv")
test  = pd.read_csv(f"{DATA_DIR}/dataset_B_testing.csv")
sub   = pd.read_csv(f"{DATA_DIR}/dataset_B_example_submission.csv")

print(f"train {train.shape} | test {test.shape} | example submission {sub.shape}")

# --- structural checks (fail loudly rather than silently continue) ---
assert TARGET in train.columns and TARGET not in test.columns
assert set(train.columns) - set(test.columns) == {TARGET}, "train/test feature mismatch"
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert len(set(train[ID_COL]) & set(test[ID_COL])) == 0, "ID overlap between train and test"
assert list(sub.columns) == [ID_COL, TARGET]
assert (sub[ID_COL].values == test[ID_COL].values).all(), "submission IDs/order differ from test"
assert set(train[TARGET].unique()) <= {0, 1}, "unexpected target encoding"
print("All integrity checks passed.")

print("\nTarget distribution (train):")
print(train[TARGET].value_counts().rename("count").to_frame().assign(share=lambda d: (d["count"]/len(train)).round(4)))
print("\nExample-submission values (a placeholder, NOT a valid answer):", sub[TARGET].unique())
print("Exact duplicate feature rows in train:", train.drop(columns=[ID_COL]).duplicated().sum())

### 1.3 Feature groups (from `Dataset_B_Information.pdf`, checked against the CSV header)
Note: the `opinion_*` variables use `3 = Don't know`, which sits in the middle of the numeric scale but is **not** a true ordinal midpoint – we will test treating it as its own category in Phase 3.

In [ ]:
BINARY  = ["behavioral_antiviral_meds","behavioral_avoidance","behavioral_face_mask","behavioral_wash_hands",
           "behavioral_large_gatherings","behavioral_outside_home","behavioral_touch_face",
           "doctor_recc_h1n1","child_under_6_months","health_worker","health_insurance"]
ORDINAL = ["h1n1_concern","h1n1_knowledge","opinion_h1n1_vacc_effective","opinion_h1n1_risk",
           "opinion_h1n1_sick_from_vacc","age_group","education","income_poverty",
           "household_adults","household_children","number_chronic_med_conditions"]
NOMINAL = ["race","sex","marital_status","rent_or_own","employment_status","census_msa","employment_sector"]

features = [c for c in train.columns if c not in (ID_COL, TARGET)]
assert sorted(BINARY + ORDINAL + NOMINAL) == sorted(features), "feature groups do not cover the CSV columns"
print(f"{len(features)} features = {len(BINARY)} binary + {len(ORDINAL)} ordinal + {len(NOMINAL)} nominal")

# Observed category values for the text columns (watch for odd spellings, e.g. a double space in a census_msa label)
for c in ["age_group","education","income_poverty","census_msa","employment_status"]:
    print(f"\n{c}: {sorted(train[c].dropna().unique().tolist())}")

# Train vs test missingness – a first look for distribution shift (investigated properly in Phase 2)
miss = pd.DataFrame({"train_%": train[features].isna().mean()*100,
                     "test_%":  test[features].isna().mean()*100}).round(2)
print("\nMissingness (%), top 10 by train:"); print(miss.sort_values("train_%", ascending=False).head(10))

### 1.4 Cost-based evaluation framework (⚠ PENDING CONFIRMATION)
The brief gives only two numbers: vaccine = **$500**, hospitalisation = **$25,000**. It does not state which error carries which cost,
so the cost matrix below is an **explicit, switchable assumption**. Expected-cost-optimal rule:
predict 1 iff `p > (c_FP − c_TN) / ((c_FP − c_TN) + (c_FN − c_TP))`.

In [ ]:
# Cost matrix in dollars per respondent, keyed by (actual, predicted).
SCENARIOS = {
  # A: predicting "vaccinated" for someone who is NOT leaves them unprotected -> hospitalisation risk ($25k);
  #    predicting "not vaccinated" for someone who IS wastes a $500 intervention.
  "A_FP25000_FN500": dict(TP=0, TN=0, FP=25_000, FN=500),
  # B: the reverse reading.
  "B_FP500_FN25000": dict(TP=0, TN=0, FP=500, FN=25_000),
  # C: we vaccinate ($500) everyone predicted 0; FP (predicted 1, actually 0) = missed -> $25k.
  "C_vaccinate_predicted0": dict(TP=0, TN=500, FP=25_000, FN=500),
}
ACTIVE_SCENARIO = "A_FP25000_FN500"      # <-- change once the metric is confirmed

def total_cost(y_true, y_pred, cost=None):
    """Total dollar cost of hard 0/1 predictions under a cost matrix."""
    cost = cost or SCENARIOS[ACTIVE_SCENARIO]
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = ((y_true==1)&(y_pred==1)).sum(); tn = ((y_true==0)&(y_pred==0)).sum()
    fp = ((y_true==0)&(y_pred==1)).sum(); fn = ((y_true==1)&(y_pred==0)).sum()
    return tp*cost["TP"] + tn*cost["TN"] + fp*cost["FP"] + fn*cost["FN"]

def theoretical_threshold(cost=None):
    """Probability above which predicting 1 minimises expected cost (assumes calibrated probabilities)."""
    cost = cost or SCENARIOS[ACTIVE_SCENARIO]
    a, b = cost["FP"]-cost["TN"], cost["FN"]-cost["TP"]
    return a/(a+b)

# Descriptive reference points on the training labels (trivial strategies, NOT model results)
y = train[TARGET].values
rows = []
for name, cm in SCENARIOS.items():
    rows.append(dict(scenario=name, theory_threshold=round(theoretical_threshold(cm),4),
                     cost_all_predict_0=total_cost(y, np.zeros_like(y), cm),
                     cost_all_predict_1=total_cost(y, np.ones_like(y), cm)))
print(pd.DataFrame(rows).to_string(index=False))

### 1.5 Experiment tracker (used in every later phase)

In [ ]:
EXPERIMENTS = []
def log_experiment(name, phase, model, preprocessing, cv_scheme, metrics: dict, params: dict = None, notes=""):
    """Append one experiment to the in-memory table and persist it to outputs/experiment_log.csv."""
    EXPERIMENTS.append(dict(name=name, phase=phase, model=model, preprocessing=preprocessing, cv_scheme=cv_scheme,
                            params=json.dumps(params or {}, default=str), notes=notes, **metrics))
    df = pd.DataFrame(EXPERIMENTS); df.to_csv(f"{OUT_DIR}/experiment_log.csv", index=False)
    return df

# Save the Phase 1 configuration so the team has one source of truth
config = dict(seed=SEED, target=TARGET, id_col=ID_COL, binary=BINARY, ordinal=ORDINAL, nominal=NOMINAL,
              scenarios=SCENARIOS, active_scenario=ACTIVE_SCENARIO)
json.dump(config, open(f"{OUT_DIR}/config_phase1.json","w"), indent=2)
print("Saved outputs/config_phase1.json; experiment log ready.")